# Projet consommation — mêmes données, six jalons

Ouvrir ce notebook dans le dossier Projet_IA pour importer correction.py et retrouver data/consommation.csv.
Les données sont simulées. Les partitions chronologiques sont fixées dès le départ.
Les fonctions importées dans la correction sont expliquées dans les six chapitres avancés.


In [ ]:
from pathlib import Path
import numpy as np
from correction import (charger, ajuster_echelle, ajouter_biais, differences_finies,
                        cout_gradient, descente, ridge, rmse,
                        entrainer_classification, sigmoide)
X, y, (train, validation, test) = charger()
print(X.shape, y.shape)


## Jalon 1 — Géométrie

Relire les hypothèses et critères de validation dans README.md.

In [ ]:
echelles = np.array([10., 1., 1.])
profils, q = X[train] / echelles, X[validation][0] / echelles
normes = np.linalg.norm(profils, axis=1)
valides = normes > 1e-12
scores = np.full(len(profils), -np.inf)
scores[valides] = profils[valides] @ q / (normes[valides] * np.linalg.norm(q))
voisins = np.argsort(scores)[-3:][::-1]
print(voisins + 1, scores[voisins])


## Jalon 2 — Moindres carrés

Relire les hypothèses et critères de validation dans README.md.

In [ ]:
A_brut = ajouter_biais(X)
theta_brut, _, rang, _ = np.linalg.lstsq(A_brut[train], y[train], rcond=None)
r = y[train] - A_brut[train] @ theta_brut
print(rang, A_brut[train].T @ r)
print("RMSE validation", rmse(A_brut[validation] @ theta_brut, y[validation]))


## Jalon 3 — Standardisation et SVD

Relire les hypothèses et critères de validation dans README.md.

In [ ]:
mu, sigma = ajuster_echelle(X[train])
Z = (X - mu) / sigma
A = ajouter_biais(Z)
U, s, Vt = np.linalg.svd(Z[train], full_matrices=False)
parts = s**2 / np.sum(s**2)
k = int(np.searchsorted(np.cumsum(parts), .95) + 1)
print(parts, k)
reconstruction = (Z[validation] @ Vt[:k].T) @ Vt[:k]
print(np.mean((Z[validation] - reconstruction)**2))


## Jalon 4 — Vérifier le gradient

Relire les hypothèses et critères de validation dans README.md.

In [ ]:
theta0 = np.zeros(A.shape[1])
valeur, analytique = cout_gradient(A[train], y[train], theta0)
numerique = differences_finies(lambda t: cout_gradient(A[train], y[train], t)[0], theta0)
print(valeur, np.max(np.abs(analytique - numerique)))


## Jalon 5 — Optimisation et choix sur validation

Relire les hypothèses et critères de validation dans README.md.

In [ ]:
candidats = [0., .001, .01, .1, 1.]
evaluations = [(rmse(A[validation] @ ridge(A[train], y[train], l), y[validation]), l) for l in candidats]
_, penalite = min(evaluations)
theta, historique, pas = descente(A[train], y[train], penalite)
print("lambda", penalite, "pas", pas, "perte finale", historique[-1])
# Aucun accès au test pendant la sélection.


## Jalon 6 — Classification et évaluation finale

Relire les hypothèses et critères de validation dans README.md.

In [ ]:
seuil = np.median(y[train])
classes = (y > seuil).astype(float)
logistique, reseau = entrainer_classification(Z[train], classes[train], Z[validation], classes[validation])
# Tous les choix sont désormais figés : première évaluation du test.
pred_l = sigmoide(A[test] @ logistique[1]) >= .5
w1, b1, w2, b2 = reseau[1]
pred_r = sigmoide((np.tanh(Z[test] @ w1 + b1) @ w2 + b2).ravel()) >= .5
print("RMSE régression test", rmse(A[test] @ theta, y[test]))
for nom, predictions in [("logistique", pred_l), ("réseau", pred_r)]:
    confusion = np.zeros((2, 2), dtype=int)
    np.add.at(confusion, (classes[test].astype(int), predictions.astype(int)), 1)
    print(nom, np.mean(predictions == classes[test]), confusion)
